# 6.12 — Dropout & DropConnect

Dropout and DropConnect regularize neural networks by training many thinned versions of the same model: dropout randomly removes activations, while DropConnect randomly removes weights. The key mathematical trick is **inverted scaling**: divide the kept signal by the keep probability so the expected forward signal stays on the same scale during training and evaluation.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Dropout and DropConnect one idea at a time. Run each cell in order and read the printed intermediate values — the mask, expectation, gradient flow, and weight masking are all made visible. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, masks, random draws, and linear algebra.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for Bernoulli masks.

### 1. Activations are signals we can randomly gate

A layer produces an activation vector `h`. Dropout does not change the formula that produced `h`; it inserts a random gate after it. Each coordinate either survives or becomes zero, so no downstream unit can assume every upstream feature will always be present.

In [ ]:
h_w = np.array([2.0, -1.0, 0.5, 3.0])  # four activations from a toy hidden layer.
q_w = 0.5  # keep probability: each coordinate survives with probability 1/2.
mask_w = np.array([1.0, 0.0, 1.0, 0.0])  # one visible Bernoulli(q) draw.
gated_w = mask_w * h_w  # ordinary dropout gate before any scaling.
print("h:", h_w)
print("mask:", mask_w)
print("masked h:", gated_w)

▶ What you'll see: coordinates 1 and 3 are removed, while coordinates 0 and 2 remain.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(h_w)) - 0.18, h_w, width=0.36, label="original h", color="gray")
plt.bar(np.arange(len(h_w)) + 0.18, gated_w, width=0.36, label="masked h", color="crimson")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: dropout gates activations")
plt.xlabel("activation coordinate")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: the red bars are a thinned version of the gray bars; removed units are exactly zero.

*Why it's done this way:* a hidden unit can over-specialize if it always sees the same partners. Multiplying by Bernoulli gates forces the next layer to solve the task with many partial subnetworks, so useful features must remain useful even when other features disappear.

### 2. Inverted dropout preserves the expected activation scale

If we only multiply by a 0/1 mask, the average signal shrinks by the keep probability `q`. Inverted dropout fixes that during training by using

$$\tilde h = \frac{m\odot h}{q},\qquad m_i\sim\mathrm{Bernoulli}(q).$$

Since $\mathbb E[m_i]=q$, the expectation is $\mathbb E[m_i h_i/q]=h_i$. That is why the layer can use the identity map at inference time.

In [ ]:
h_w = np.array([2.0, -1.0, 0.5, 3.0])
q_w = 0.5
masks_w = (np.random.rand(8, h_w.size) < q_w).astype(float)  # eight Bernoulli masks.
plain_mean_w = (masks_w * h_w).mean(axis=0)  # unscaled dropout average.
inverted_mean_w = (masks_w * h_w / q_w).mean(axis=0)  # inverted dropout average.
print("plain dropout mean:", np.round(plain_mean_w, 3))
print("inverted dropout mean:", np.round(inverted_mean_w, 3))
print("original h:", h_w)

▶ What you'll see: with only eight masks the estimates are noisy, but inverted dropout is centered near the original scale instead of half scale.

In [ ]:
np.random.seed(1)
many_masks_w = (np.random.rand(20000, h_w.size) < q_w).astype(float)
mc_mean_w = (many_masks_w * h_w / q_w).mean(axis=0)
print("Monte Carlo E[m*h/q]:", np.round(mc_mean_w, 3))
assert np.allclose(mc_mean_w, h_w, atol=0.04)

▶ What you'll see: the Monte Carlo average is essentially the original activation vector.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(h_w)) - 0.25, h_w, width=0.25, label="h", color="black")
plt.bar(np.arange(len(h_w)), (many_masks_w * h_w).mean(axis=0), width=0.25, label="E[mh]", color="orange")
plt.bar(np.arange(len(h_w)) + 0.25, mc_mean_w, width=0.25, label="E[mh/q]", color="seagreen")
plt.title("2: inverted scaling preserves expectation")
plt.xlabel("coordinate")
plt.ylabel("average value")
plt.legend()
plt.show()

▶ What you'll see: the unscaled average shrinks toward `q*h`, while inverted dropout lines up with `h`.

*Why it's done this way:* the division by `q` is not arbitrary inflation. It exactly cancels the expected fraction of units removed, so training-time activations and inference-time activations live on the same numerical scale.

### 3. Train-time randomness becomes inference-time determinism

With inverted dropout, the model is noisy while learning and deterministic while evaluating. Training samples one mask at a time; inference uses the full activation vector with no mask and no extra multiplier. That keeps predictions stable without changing the expected signal learned during training.

In [ ]:
np.random.seed(2)
x_w = np.array([1.5, -0.5])
W_w = np.array([[1.4, -0.7], [0.3, 0.8], [-0.6, 1.1]])
b_w = np.array([0.3, 0.1, -0.2])
z_w = W_w @ x_w + b_w
h_w = np.maximum(0, z_w)
print("pre-activation z:", np.round(z_w, 3))
print("ReLU h:", np.round(h_w, 3))
assert round(float(z_w[0]), 3) == 2.75

▶ What you'll see: the first ReLU activation is the visible `2.750` scratch-pass signal from the lesson content.

In [ ]:
q_w = 0.6
train_outputs_w = []
for _ in range(12):
    mask_w = (np.random.rand(h_w.size) < q_w).astype(float)
    train_outputs_w.append(mask_w * h_w / q_w)
train_outputs_w = np.array(train_outputs_w)
print("first three train samples:\n", np.round(train_outputs_w[:3], 3))
print("inference output:", np.round(h_w, 3))

▶ What you'll see: training outputs jump around because masks differ, but inference is the original ReLU vector.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(train_outputs_w[:, 0], marker="o", label="train samples: coord 0", color="crimson")
plt.axhline(h_w[0], color="black", linestyle="--", label="inference value")
plt.title("3: noisy training, deterministic inference")
plt.xlabel("mask draw")
plt.ylabel("activation value")
plt.legend()
plt.show()

▶ What you'll see: sampled train-time values are either 0 or `h/q`, while the dashed inference line is stable.

*Why it's done this way:* training needs perturbations to discourage co-adaptation, but evaluation needs repeatable predictions. Inverted scaling lets us remove the stochastic mask at inference without manually multiplying by `q` later.

### 4. The same mask gates gradients backward

Dropout affects backpropagation too. If a coordinate was dropped in the forward pass, its local output was zero, so the gradient through that coordinate is also zero. Kept coordinates receive an amplified gradient by the same `1/q` factor used in the forward pass.

In [ ]:
h_w = np.array([2.0, -1.0, 0.5, 3.0])
q_w = 0.5
mask_w = np.array([1.0, 0.0, 1.0, 0.0])
upstream_grad_w = np.array([0.2, -0.4, 1.0, 0.3])
grad_h_w = upstream_grad_w * mask_w / q_w
print("upstream grad:", upstream_grad_w)
print("gradient through dropout:", grad_h_w)

▶ What you'll see: dropped coordinates receive exactly zero gradient; kept ones double because `q=0.5`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(4) - 0.18, upstream_grad_w, width=0.36, label="incoming", color="gray")
plt.bar(np.arange(4) + 0.18, grad_h_w, width=0.36, label="after dropout", color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("4: dropout gates backward signal")
plt.xlabel("coordinate")
plt.ylabel("gradient")
plt.legend()
plt.show()

▶ What you'll see: the backward signal is thinned in the exact coordinates that were removed forward.

*Why it's done this way:* backprop applies the chain rule to `m*h/q`. The derivative with respect to `h` is `m/q`, so the same random subnetwork that made the forward prediction is the one that receives the update.

### 5. DropConnect masks weights instead of activations

Dropout removes entries of the activation vector. DropConnect removes entries of the weight matrix itself. A unit may still receive all inputs, but some connections are missing for that training example, so the affine map changes randomly.

In [ ]:
x_w = np.array([1.0, -2.0, 0.5])
W_w = np.array([[0.8, -0.4, 0.2], [1.2, 0.5, -0.7]])
b_w = np.array([0.1, -0.2])
q_w = 0.5
maskW_w = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
full_y_w = W_w @ x_w + b_w
dropconnect_y_w = (maskW_w * W_w / q_w) @ x_w + b_w
print("full affine:", np.round(full_y_w, 3))
print("DropConnect affine:", np.round(dropconnect_y_w, 3))

▶ What you'll see: masking weights changes the affine output even though the input vector is unchanged.

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(maskW_w, cmap="Greys", vmin=0, vmax=1)
plt.colorbar(label="kept connection")
plt.title("5: DropConnect weight mask")
plt.xlabel("input coordinate")
plt.ylabel("output unit")
plt.show()

▶ What you'll see: white/black cells show which individual connections are kept or removed.

In [ ]:
np.random.seed(3)
mc_y_w = []
for _ in range(20000):
    mW_w = (np.random.rand(*W_w.shape) < q_w).astype(float)
    mc_y_w.append((mW_w * W_w / q_w) @ x_w + b_w)
mc_y_w = np.array(mc_y_w).mean(axis=0)
print("Monte Carlo E[DropConnect y]:", np.round(mc_y_w, 3))
print("full y:", np.round(full_y_w, 3))
assert np.allclose(mc_y_w, full_y_w, atol=0.03)

▶ What you'll see: inverted DropConnect is also expectation-preserving for the affine output.

*Why it's done this way:* masking weights regularizes connections rather than units. The expectation argument is the same as dropout because each kept weight contributes `W_ij/q` with probability `q`, giving expected contribution `W_ij`.

### 6. Random thinning acts like an ensemble of subnetworks

Each dropout mask selects a different subnetwork. A tiny layer with four dropout gates has $2^4=16$ possible activation subnetworks. Training shares weights across all of them, which approximates averaging many related models without explicitly storing each one.

In [ ]:
n_units_w = 4
n_subnets_w = 2 ** n_units_w
print("possible masks for 4 units:", n_subnets_w)
all_masks_w = np.array([[int(bit) for bit in format(k, "04b")] for k in range(n_subnets_w)])
print("first five masks:\n", all_masks_w[:5])
assert n_subnets_w == 16

▶ What you'll see: even four units create sixteen possible thinned subnetworks.

In [ ]:
q_w = 0.5
probs_w = (q_w ** all_masks_w.sum(axis=1)) * ((1 - q_w) ** (n_units_w - all_masks_w.sum(axis=1)))
counts_w = np.bincount(all_masks_w.sum(axis=1), minlength=n_units_w + 1)
print("mask count by kept units:", counts_w)
print("probability mass sums to:", round(float(probs_w.sum()), 3))
assert round(float(probs_w.sum()), 3) == 1.0

▶ What you'll see: masks with two kept units are most numerous when `q=0.5`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(n_units_w + 1), counts_w, color="mediumpurple")
plt.title("6: number of subnetworks by kept units")
plt.xlabel("kept activation count")
plt.ylabel("number of masks")
plt.show()

▶ What you'll see: the subnetwork count follows the familiar binomial shape.

*Why it's done this way:* the ensemble view explains why dropout can reduce overfitting: many subnetworks must agree through shared weights. The deterministic inference network is a cheap approximation to the average prediction of those thinned networks.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, masks, random draws, and scratch neural-network arithmetic.
import matplotlib.pyplot as plt  # load Matplotlib for small visual checks of masks, activations, losses, and weights.
np.random.seed(0)  # make stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Create a Bernoulli keep mask

**Goal.** Draw a dropout mask for an activation vector, because dropout starts by deciding which units survive this training pass. We build it in 2 steps.

In [ ]:
h_b1 = np.array([1.0, 2.0, -1.0, 0.5])  # define a toy activation vector.
q_b1 = 0.5  # keep each coordinate with probability one half.
np.random.seed(1)  # seed this local draw so the example is repeatable.
mask_b1 = (np.random.rand(h_b1.size) < q_b1).astype(float)  # draw Bernoulli(q) gates.
print("activation h:", h_b1)
print("mask:", mask_b1)

▶ What you'll see: a 0/1 vector with the same shape as the activation.

In [ ]:
kept_b1 = int(mask_b1.sum())  # count surviving units.
print("kept units:", kept_b1, "of", h_b1.size)
assert mask_b1.shape == h_b1.shape
plt.figure(figsize=(4, 3))
plt.bar(range(h_b1.size), mask_b1, color="seagreen")
plt.title("Basic 1: Bernoulli keep mask")
plt.xlabel("unit")
plt.ylabel("kept = 1")
plt.ylim(0, 1.2)
plt.show()

▶ What you'll see: bars at 1 are kept units and bars at 0 are dropped units.

👀 Takeaway: dropout uses one Bernoulli random variable per activation coordinate.

### Basic 2 — Apply plain dropout

**Goal.** Multiply activations by a mask, because dropped units should contribute exactly zero to the next layer. We build it in 2 steps.

In [ ]:
h_b2 = np.array([1.0, 2.0, -1.0, 0.5])  # define the activation vector.
mask_b2 = np.array([1.0, 0.0, 1.0, 0.0])  # choose a visible mask for inspection.
plain_b2 = mask_b2 * h_b2  # apply ordinary unscaled dropout.
print("plain dropout output:", plain_b2)

▶ What you'll see: masked coordinates become exactly zero.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(4) - 0.18, h_b2, width=0.36, label="h", color="gray")
plt.bar(np.arange(4) + 0.18, plain_b2, width=0.36, label="m*h", color="crimson")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 2: plain dropout")
plt.legend()
plt.show()

▶ What you'll see: the masked output is a sparse version of the original activation.

👀 Takeaway: plain dropout gates information, but it also shrinks the average signal.

### Basic 3 — Apply inverted dropout

**Goal.** Divide kept activations by the keep probability, because the expected output should match the original activation scale. We build it in 2 steps.

In [ ]:
h_b3 = np.array([1.0, 2.0, -1.0, 0.5])  # define a toy activation vector.
mask_b3 = np.array([1.0, 0.0, 1.0, 0.0])  # keep two coordinates.
q_b3 = 0.5  # keep probability used to scale surviving units.
inverted_b3 = mask_b3 * h_b3 / q_b3  # apply inverted dropout.
print("inverted dropout output:", inverted_b3)
assert np.allclose(inverted_b3, [2.0, 0.0, -2.0, 0.0])

▶ What you'll see: kept activations double because only half are expected to survive.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["h0", "h1", "h2", "h3"], inverted_b3, color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 3: kept units scaled by 1/q")
plt.show()

▶ What you'll see: nonzero bars are larger in magnitude than the original kept activations.

👀 Takeaway: inverted dropout is `mask * h / q`, not just `mask * h`.

### Basic 4 — Check expectation with many masks

**Goal.** Verify the formula $\mathbb E[mh/q]=h$, because expectation preservation is the reason inference can skip dropout. We build it in 3 steps.

In [ ]:
h_b4 = np.array([2.0, -1.0, 0.5])  # choose activations with mixed signs.
q_b4 = 0.8  # keep probability.
np.random.seed(4)  # seed the Monte Carlo mask draws.
masks_b4 = (np.random.rand(10000, h_b4.size) < q_b4).astype(float)  # draw many masks.
print("mask matrix shape:", masks_b4.shape)

▶ What you'll see: each row is one independent dropout mask.

In [ ]:
mean_b4 = (masks_b4 * h_b4 / q_b4).mean(axis=0)  # average inverted-dropout outputs.
print("Monte Carlo mean:", np.round(mean_b4, 3))
print("original h:", h_b4)
assert np.allclose(mean_b4, h_b4, atol=0.04)

▶ What you'll see: the sample mean is close to the original vector.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(3) - 0.18, h_b4, width=0.36, label="h", color="black")
plt.bar(np.arange(3) + 0.18, mean_b4, width=0.36, label="mean dropout", color="seagreen")
plt.title("Basic 4: expectation check")
plt.legend()
plt.show()

▶ What you'll see: paired bars nearly overlap coordinate by coordinate.

👀 Takeaway: inverted dropout preserves the expected activation, not each individual sample.

### Basic 5 — Compare train and inference outputs

**Goal.** See that training is stochastic and inference is deterministic, because dropout masks are used only while learning. We build it in 2 steps.

In [ ]:
h_b5 = np.array([3.0, 1.0, 2.0])  # define deterministic inference activations.
q_b5 = 0.5  # keep probability during training.
np.random.seed(5)  # seed the train-time mask.
mask_b5 = (np.random.rand(h_b5.size) < q_b5).astype(float)  # draw one training mask.
train_b5 = mask_b5 * h_b5 / q_b5  # stochastic train-time output.
infer_b5 = h_b5.copy()  # deterministic inference output for inverted dropout.
print("train output:", train_b5)
print("inference output:", infer_b5)

▶ What you'll see: train output has zeros and scaled survivors, while inference keeps all activations.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(3) - 0.18, train_b5, width=0.36, label="train", color="crimson")
plt.bar(np.arange(3) + 0.18, infer_b5, width=0.36, label="inference", color="gray")
plt.title("Basic 5: train vs inference")
plt.legend()
plt.show()

▶ What you'll see: the inference bars are stable and unmasked.

👀 Takeaway: inverted dropout moves the scaling burden into training so inference is just the full network.

### Basic 6 — Dropout after ReLU

**Goal.** Apply dropout after a ReLU hidden layer, because dropout usually regularizes activations produced by nonlinear units. We build it in 3 steps.

In [ ]:
x_b6 = np.array([1.5, -0.5])  # two inputs from the lesson scratch pass.
w_b6 = np.array([1.4, -0.7])  # one neuron's weights.
b_b6 = 0.3  # bias term.
z_b6 = float(w_b6 @ x_b6 + b_b6)  # affine signal before the nonlinearity.
h_b6 = max(0.0, z_b6)  # ReLU activation.
print("z:", round(z_b6, 3), "ReLU h:", round(h_b6, 3))
assert round(z_b6, 3) == 2.75

▶ What you'll see: the visible affine calculation produces a positive ReLU activation of 2.750.

In [ ]:
q_b6 = 0.5
mask_b6 = 1.0  # this particular draw keeps the unit.
dropped_b6 = mask_b6 * h_b6 / q_b6
print("dropout output:", round(dropped_b6, 3))
assert round(dropped_b6, 3) == 5.5

▶ What you'll see: the kept ReLU activation doubles when `q=0.5`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["z", "ReLU", "dropout"], [z_b6, h_b6, dropped_b6], color=["gray", "orange", "teal"])
plt.title("Basic 6: affine → ReLU → dropout")
plt.ylabel("signal")
plt.show()

▶ What you'll see: dropout is applied after the activation has already been shaped by ReLU.

👀 Takeaway: dropout regularizes the hidden representation that later layers consume.

### Basic 7 — Backpropagate through a dropout mask

**Goal.** Gate a gradient with the same mask, because the chain rule uses the derivative of `m*h/q`. We build it in 2 steps.

In [ ]:
g_b7 = np.array([0.3, -0.2, 0.8])  # incoming gradient from the next layer.
mask_b7 = np.array([1.0, 0.0, 1.0])  # forward-pass dropout mask.
q_b7 = 0.5  # keep probability.
grad_b7 = g_b7 * mask_b7 / q_b7  # local gradient with respect to h.
print("incoming gradient:", g_b7)
print("gradient after dropout:", grad_b7)

▶ What you'll see: dropped coordinate 1 receives zero gradient.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(3) - 0.18, g_b7, width=0.36, label="incoming", color="gray")
plt.bar(np.arange(3) + 0.18, grad_b7, width=0.36, label="after mask", color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 7: gradient gating")
plt.legend()
plt.show()

▶ What you'll see: kept gradients are scaled, and dropped gradients vanish.

👀 Takeaway: dropout chooses which units participate in both prediction and learning for that pass.

### Basic 8 — Mask weights with DropConnect

**Goal.** Apply a Bernoulli mask to a weight matrix, because DropConnect regularizes individual connections instead of activations. We build it in 2 steps.

In [ ]:
W_b8 = np.array([[0.5, -1.0], [1.5, 0.2]])  # two-output, two-input weight matrix.
mask_b8 = np.array([[1.0, 0.0], [0.0, 1.0]])  # keep a diagonal set of connections.
q_b8 = 0.5  # keep probability.
W_drop_b8 = mask_b8 * W_b8 / q_b8  # inverted DropConnect weights.
print("masked weights:\n", W_drop_b8)

▶ What you'll see: dropped connections become zero, while kept weights are divided by `q`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(W_drop_b8, cmap="coolwarm")
plt.colorbar(label="effective weight")
plt.title("Basic 8: DropConnect weight matrix")
plt.xlabel("input")
plt.ylabel("output")
plt.show()

▶ What you'll see: a heatmap of the effective training-time weight matrix.

👀 Takeaway: DropConnect perturbs parameters used by the affine map, not the activation vector after it.

### Basic 9 — Compute a DropConnect forward pass

**Goal.** Compare a full affine output to a masked-weight output, because DropConnect changes the linear map for each training example. We build it in 2 steps.

In [ ]:
x_b9 = np.array([1.0, -2.0])  # input vector.
W_b9 = np.array([[0.5, -1.0], [1.5, 0.2]])  # full weights.
b_b9 = np.array([0.1, -0.1])  # bias vector.
full_b9 = W_b9 @ x_b9 + b_b9  # ordinary affine output.
print("full output:", np.round(full_b9, 3))

▶ What you'll see: the baseline affine output before any random connection removal.

In [ ]:
mask_b9 = np.array([[1.0, 0.0], [0.0, 1.0]])
q_b9 = 0.5
dc_b9 = (mask_b9 * W_b9 / q_b9) @ x_b9 + b_b9
print("DropConnect output:", np.round(dc_b9, 3))
assert np.allclose(np.round(full_b9, 3), [2.6, 1.0])
plt.figure(figsize=(4, 3))
plt.bar(["full y0", "full y1", "DC y0", "DC y1"], [full_b9[0], full_b9[1], dc_b9[0], dc_b9[1]], color=["gray", "gray", "purple", "purple"])
plt.title("Basic 9: full vs DropConnect affine")
plt.xticks(rotation=20)
plt.show()

▶ What you'll see: DropConnect changes both output coordinates because the effective weights changed.

👀 Takeaway: DropConnect injects noise before the nonlinearity by changing which weights are active.

### Basic 10 — Count possible dropout subnetworks

**Goal.** Count masks, because dropout can be understood as training many thinned subnetworks with shared parameters. We build it in 2 steps.

In [ ]:
n_b10 = 5  # number of dropout-controlled units.
subnets_b10 = 2 ** n_b10  # every unit can be on or off.
print("possible subnetworks:", subnets_b10)
assert subnets_b10 == 32

▶ What you'll see: five gated units already produce 32 possible masks.

In [ ]:
all_masks_b10 = np.array([[int(bit_b10) for bit_b10 in format(i_b10, "05b")] for i_b10 in range(subnets_b10)])  # enumerate every 5-bit mask.
counts_b10 = np.bincount(all_masks_b10.sum(axis=1), minlength=n_b10 + 1)  # count masks by number of kept units.
print("counts by kept units:", counts_b10)
plt.figure(figsize=(4, 3))
plt.bar(range(n_b10 + 1), counts_b10, color="mediumpurple")
plt.title("Basic 10: subnetworks by kept units")
plt.xlabel("kept units")
plt.ylabel("mask count")
plt.show()

▶ What you'll see: the number of subnetworks is largest around half the units kept.

👀 Takeaway: random masks implicitly expose the same weights to many related network architectures.

## 🟡 Easy

### Easy 1 — Simulate dropout variance versus keep probability

**Goal.** Measure how output noise changes as `q` changes, because smaller keep probability means more aggressive regularization and larger variance. We build it in 3 steps.

In [ ]:
h_e1 = np.array([2.0, -1.0, 0.5])  # fixed activation vector.
qs_e1 = np.array([0.2, 0.5, 0.8])  # keep probabilities to compare.
np.random.seed(11)  # seed the simulation.
print("keep probabilities:", qs_e1)

▶ What you'll see: the sweep goes from aggressive dropout to mild dropout.

In [ ]:
variances_e1 = []
for q_e1 in qs_e1:
    masks_e1 = (np.random.rand(12000, h_e1.size) < q_e1).astype(float)
    outs_e1 = masks_e1 * h_e1 / q_e1
    variances_e1.append(float(outs_e1[:, 0].var()))
variances_e1 = np.array(variances_e1)
print("variance of coordinate 0:", np.round(variances_e1, 3))
assert variances_e1[0] > variances_e1[1] > variances_e1[2]

▶ What you'll see: the variance is highest for `q=0.2` and lowest for `q=0.8`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(qs_e1, variances_e1, marker="o", color="crimson")
plt.title("Easy 1: dropout variance vs keep probability")
plt.xlabel("keep probability q")
plt.ylabel("Var[masked h0/q]")
plt.show()

▶ What you'll see: increasing `q` reduces training-time noise.

👀 Takeaway: dropout strength is a bias-variance knob; lower `q` perturbs the representation more strongly.

### Easy 2 — Compare inverted and non-inverted inference

**Goal.** Show why inverted dropout avoids an inference-time multiplier, because ordinary dropout would require scaling the full network by `q` at evaluation. We build it in 3 steps.

In [ ]:
h_e2 = np.array([4.0, 2.0, 1.0])  # deterministic hidden activation.
q_e2 = 0.5  # keep probability.
np.random.seed(12)
masks_e2 = (np.random.rand(20000, h_e2.size) < q_e2).astype(float)
print("original h:", h_e2)

▶ What you'll see: this is the inference vector we want to preserve in expectation.

In [ ]:
plain_train_mean_e2 = (masks_e2 * h_e2).mean(axis=0)
inverted_train_mean_e2 = (masks_e2 * h_e2 / q_e2).mean(axis=0)
plain_infer_e2 = q_e2 * h_e2
inverted_infer_e2 = h_e2
print("plain train mean:", np.round(plain_train_mean_e2, 3), "plain inference scaling:", plain_infer_e2)
print("inverted train mean:", np.round(inverted_train_mean_e2, 3), "inverted inference:", inverted_infer_e2)
assert np.allclose(inverted_train_mean_e2, inverted_infer_e2, atol=0.04)

▶ What you'll see: ordinary dropout's train mean matches `q*h`, while inverted dropout's train mean matches `h`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["plain train", "plain infer", "inverted train", "inverted infer"], [plain_train_mean_e2[0], plain_infer_e2[0], inverted_train_mean_e2[0], inverted_infer_e2[0]], color=["orange", "orange", "teal", "teal"])
plt.title("Easy 2: where the scaling lives")
plt.ylabel("coordinate 0 value")
plt.xticks(rotation=20)
plt.show()

▶ What you'll see: inverted dropout aligns train expectation with inference directly.

👀 Takeaway: inverted dropout is preferred because inference is the unmodified full network.

### Easy 3 — Train a tiny linear model with dropout on inputs

**Goal.** Fit a two-feature regression model while randomly dropping input features, because input dropout forces weights to be useful under missing-feature perturbations. We build it in 4 steps.

In [ ]:
np.random.seed(13)
X_e3 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [1.0, 2.0]])
y_e3 = np.array([1.0, -1.0, 0.0, 1.0, -1.0])
w_e3 = np.zeros(2)
print("X shape:", X_e3.shape, "initial w:", w_e3)

▶ What you'll see: a tiny supervised dataset and zero-initialized weights.

In [ ]:
q_e3 = 0.8
lr_e3 = 0.05
losses_e3 = []
for epoch_e3 in range(300):
    pred_full_e3 = X_e3 @ w_e3
    losses_e3.append(float(np.mean((pred_full_e3 - y_e3) ** 2)))
    for i_e3 in range(X_e3.shape[0]):
        mask_e3 = (np.random.rand(X_e3.shape[1]) < q_e3).astype(float)
        x_drop_e3 = X_e3[i_e3] * mask_e3 / q_e3
        err_e3 = float(x_drop_e3 @ w_e3 - y_e3[i_e3])
        w_e3 -= lr_e3 * err_e3 * x_drop_e3
print("learned w:", np.round(w_e3, 3), "loss:", round(losses_e3[-1], 3))
assert losses_e3[-1] < losses_e3[0]

▶ What you'll see: the full-data loss falls from the initial value after repeated noisy updates.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_e3, color="teal")
plt.title("Easy 3: input-dropout training curve")
plt.xlabel("epoch")
plt.ylabel("full-data MSE")
plt.show()

▶ What you'll see: the curve generally descends despite stochastic feature removal.

In [ ]:
preds_e3 = X_e3 @ w_e3
print("final predictions:", np.round(preds_e3, 3))
plt.figure(figsize=(4, 3))
plt.scatter(y_e3, preds_e3, color="purple")
plt.plot([-1.2, 1.2], [-1.2, 1.2], color="black", linestyle="--")
plt.title("Easy 3: predictions vs targets")
plt.xlabel("target")
plt.ylabel("prediction")
plt.show()

▶ What you'll see: predictions move toward the target diagonal but remain imperfect because regularization adds noise.

👀 Takeaway: dropout can be applied to inputs too; it trains the model to tolerate missing or perturbed features.

### Easy 4 — Compare dropout and DropConnect masks side by side

**Goal.** Distinguish activation masking from weight masking, because the two methods regularize different objects in the computation graph. We build it in 3 steps.

In [ ]:
x_e4 = np.array([1.0, -1.0, 2.0])
W_e4 = np.array([[0.4, -0.2, 0.6], [1.0, 0.5, -0.3]])
h_e4 = W_e4 @ x_e4
print("pre-dropout activation:", np.round(h_e4, 3))

▶ What you'll see: the same affine activation will be used for both comparisons.

In [ ]:
q_e4 = 0.5
act_mask_e4 = np.array([1.0, 0.0])
weight_mask_e4 = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
dropout_y_e4 = act_mask_e4 * h_e4 / q_e4
dropconnect_y_e4 = (weight_mask_e4 * W_e4 / q_e4) @ x_e4
print("dropout output:", np.round(dropout_y_e4, 3))
print("DropConnect output:", np.round(dropconnect_y_e4, 3))

▶ What you'll see: dropout zeros an output unit, while DropConnect recomputes both outputs with masked weights.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["full0", "full1", "dropout0", "dropout1", "DC0", "DC1"], [h_e4[0], h_e4[1], dropout_y_e4[0], dropout_y_e4[1], dropconnect_y_e4[0], dropconnect_y_e4[1]], color=["gray", "gray", "teal", "teal", "purple", "purple"])
plt.title("Easy 4: activation mask vs weight mask")
plt.xticks(rotation=25)
plt.ylabel("output")
plt.show()

▶ What you'll see: the two regularizers produce different effective forward passes.

👀 Takeaway: dropout masks units after they are computed; DropConnect masks connections before the affine sum.

### Easy 5 — Show how dropout discourages co-adaptation

**Goal.** Remove one feature at a time from a redundant representation, because dropout punishes solutions that depend on only one fragile partner. We build it in 3 steps.

In [ ]:
h_e5 = np.array([1.0, 1.0])  # two redundant features.
w_fragile_e5 = np.array([2.0, 0.0])  # relies on the first feature only.
w_shared_e5 = np.array([1.0, 1.0])  # spreads responsibility across both features.
print("fragile weights:", w_fragile_e5, "shared weights:", w_shared_e5)

▶ What you'll see: both weight vectors produce the same full-network output.

In [ ]:
masks_e5 = np.array([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
q_e5 = 0.5
fragile_out_e5 = (masks_e5 * h_e5 / q_e5) @ w_fragile_e5
shared_out_e5 = (masks_e5 * h_e5 / q_e5) @ w_shared_e5
print("fragile outputs:", fragile_out_e5)
print("shared outputs:", shared_out_e5)
assert fragile_out_e5[-1] == 0.0

▶ What you'll see: the fragile solution collapses when its one relied-upon feature is dropped.

In [ ]:
plt.figure(figsize=(5, 3))
labels_e5 = ["both kept", "only h0", "only h1"]
plt.plot(labels_e5, fragile_out_e5, marker="o", label="fragile", color="crimson")
plt.plot(labels_e5, shared_out_e5, marker="o", label="shared", color="seagreen")
plt.title("Easy 5: co-adaptation stress test")
plt.ylabel("output")
plt.xticks(rotation=15)
plt.legend()
plt.show()

▶ What you'll see: spreading responsibility makes the output less brittle across masks.

👀 Takeaway: dropout rewards representations that do not require one exact partner unit to be present.

## 🔴 Advanced

### Advanced 1 — Sweep dropout strength on a small classifier

**Goal.** Train a logistic classifier with several keep probabilities, because dropout strength changes optimization noise and regularization. We build it in 5 steps.

In [ ]:
np.random.seed(21)
X_a1 = np.vstack([np.random.normal([-1.0, -1.0], 0.5, size=(40, 2)), np.random.normal([1.0, 1.0], 0.5, size=(40, 2))])
y_a1 = np.r_[np.zeros(40), np.ones(40)]
qs_a1 = np.array([1.0, 0.8, 0.5])
print("dataset shape:", X_a1.shape, "q values:", qs_a1)

▶ What you'll see: a tiny two-cluster binary classification dataset.

In [ ]:
def sigmoid_a1(z_a1):
    return 1.0 / (1.0 + np.exp(-z_a1))

final_losses_a1 = []
weights_a1 = []
for q_a1 in qs_a1:
    np.random.seed(int(q_a1 * 100))
    w_a1 = np.zeros(2)
    b_a1 = 0.0
    for epoch_a1 in range(180):
        for i_a1 in range(X_a1.shape[0]):
            mask_a1 = (np.random.rand(2) < q_a1).astype(float) if q_a1 < 1.0 else np.ones(2)
            xd_a1 = X_a1[i_a1] * mask_a1 / q_a1
            p_a1 = sigmoid_a1(float(xd_a1 @ w_a1 + b_a1))
            err_a1 = p_a1 - y_a1[i_a1]
            w_a1 -= 0.05 * err_a1 * xd_a1
            b_a1 -= 0.05 * err_a1
    p_full_a1 = sigmoid_a1(X_a1 @ w_a1 + b_a1)
    loss_a1 = -np.mean(y_a1 * np.log(p_full_a1 + 1e-9) + (1 - y_a1) * np.log(1 - p_full_a1 + 1e-9))
    final_losses_a1.append(float(loss_a1))
    weights_a1.append(w_a1.copy())
print("final losses:", np.round(final_losses_a1, 3))
assert min(final_losses_a1) < 0.2

▶ What you'll see: all keep probabilities learn the separable toy data, but not with identical loss.

In [ ]:
norms_a1 = np.array([np.linalg.norm(w_a1) for w_a1 in weights_a1])
print("weight norms:", np.round(norms_a1, 3))
plt.figure(figsize=(5, 3))
plt.plot(qs_a1, final_losses_a1, marker="o", color="teal")
plt.title("Advanced 1: keep probability sweep")
plt.xlabel("keep probability q")
plt.ylabel("full-data log loss")
plt.show()

▶ What you'll see: a small curve showing how the chosen dropout strength affects the trained classifier.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(X_a1[:, 0], X_a1[:, 1], c=y_a1, cmap="coolwarm", alpha=0.7)
best_idx_a1 = int(np.argmin(final_losses_a1))
w_best_a1 = weights_a1[best_idx_a1]
xs_a1 = np.linspace(X_a1[:, 0].min(), X_a1[:, 0].max(), 50)
ys_a1 = -(w_best_a1[0] * xs_a1) / (w_best_a1[1] + 1e-9)
plt.plot(xs_a1, ys_a1, color="black", label=f"best q={qs_a1[best_idx_a1]}")
plt.title("Advanced 1: learned separator")
plt.legend()
plt.show()

▶ What you'll see: the best model draws a separating line between the two clusters.

👀 Takeaway: keep probability is a hyperparameter; too much noise can slow or distort learning, while too little may under-regularize.

### Advanced 2 — Measure calibration of inverted dropout by layer width

**Goal.** Show that average scale becomes more stable as many units are averaged, because independent dropout noise cancels more in wider layers. We build it in 4 steps.

In [ ]:
np.random.seed(22)
widths_a2 = np.array([4, 16, 64, 256])
q_a2 = 0.5
trials_a2 = 3000
print("widths:", widths_a2, "trials:", trials_a2)

▶ What you'll see: we will compare small and wide activation vectors.

In [ ]:
relative_errors_a2 = []
for width_a2 in widths_a2:
    h_a2 = np.ones(width_a2)
    masks_a2 = (np.random.rand(trials_a2, width_a2) < q_a2).astype(float)
    sample_means_a2 = (masks_a2 * h_a2 / q_a2).mean(axis=1)
    relative_errors_a2.append(float(np.std(sample_means_a2)))
relative_errors_a2 = np.array(relative_errors_a2)
print("std of layer-average output:", np.round(relative_errors_a2, 3))
assert relative_errors_a2[0] > relative_errors_a2[-1]

▶ What you'll see: the average output fluctuates less for wider layers.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(widths_a2, relative_errors_a2, marker="o", color="purple")
plt.xscale("log")
plt.title("Advanced 2: wider layers average dropout noise")
plt.xlabel("layer width")
plt.ylabel("std of mean activation")
plt.show()

▶ What you'll see: the curve falls as width increases.

In [ ]:
theory_a2 = np.sqrt((1 - q_a2) / (q_a2 * widths_a2))
print("theory:", np.round(theory_a2, 3))
assert np.allclose(relative_errors_a2, theory_a2, atol=0.04)
plt.figure(figsize=(5, 3))
plt.plot(widths_a2, relative_errors_a2, marker="o", label="simulation")
plt.plot(widths_a2, theory_a2, marker="s", label="sqrt((1-q)/(q n))")
plt.xscale("log")
plt.title("Advanced 2: simulation matches binomial scale")
plt.legend()
plt.show()

▶ What you'll see: simulation and theory nearly overlap.

👀 Takeaway: inverted dropout preserves expectation for every unit, and averaging many independent units reduces relative noise.

### Advanced 3 — Compare DropConnect expectation and variance by connection

**Goal.** Inspect output variance from masked weights, because DropConnect preserves expected affine output but still injects connection-level noise. We build it in 4 steps.

In [ ]:
x_a3 = np.array([1.0, -2.0, 0.5])
W_a3 = np.array([[0.8, -0.4, 0.2], [1.2, 0.5, -0.7]])
b_a3 = np.array([0.1, -0.2])
q_a3 = 0.6
full_a3 = W_a3 @ x_a3 + b_a3
print("full affine output:", np.round(full_a3, 3))

▶ What you'll see: the deterministic affine output that DropConnect should match in expectation.

In [ ]:
np.random.seed(23)
outs_a3 = []
for _ in range(20000):
    mask_a3 = (np.random.rand(*W_a3.shape) < q_a3).astype(float)
    outs_a3.append((mask_a3 * W_a3 / q_a3) @ x_a3 + b_a3)
outs_a3 = np.array(outs_a3)
mean_a3 = outs_a3.mean(axis=0)
std_a3 = outs_a3.std(axis=0)
print("DropConnect mean:", np.round(mean_a3, 3))
print("DropConnect std:", np.round(std_a3, 3))
assert np.allclose(mean_a3, full_a3, atol=0.03)

▶ What you'll see: the mean matches the full output, while standard deviations reveal stochastic spread.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(outs_a3[:, 0], bins=30, color="teal", alpha=0.8)
plt.axvline(full_a3[0], color="black", linestyle="--", label="full output")
plt.title("Advanced 3: DropConnect output distribution")
plt.xlabel("output coordinate 0")
plt.ylabel("count")
plt.legend()
plt.show()

▶ What you'll see: random connection masks create a distribution centered near the full affine value.

In [ ]:
contrib_var_a3 = ((1 - q_a3) / q_a3) * (W_a3 * x_a3[None, :]) ** 2
print("theoretical variance by output:", np.round(contrib_var_a3.sum(axis=1), 3))
assert np.allclose(std_a3 ** 2, contrib_var_a3.sum(axis=1), atol=0.04)
plt.figure(figsize=(5, 3))
plt.imshow(contrib_var_a3, cmap="magma")
plt.colorbar(label="variance contribution")
plt.title("Advanced 3: per-connection variance")
plt.xlabel("input")
plt.ylabel("output")
plt.show()

▶ What you'll see: large `W_ij*x_j` terms contribute the most DropConnect variance.

👀 Takeaway: DropConnect keeps the affine expectation but makes high-impact connections noisy during training.

### Advanced 4 — Diagnose too much dropout as underfitting

**Goal.** Compare training curves for mild and aggressive dropout, because overly small keep probabilities can make optimization noisy enough to underfit. We build it in 4 steps.

In [ ]:
np.random.seed(24)
X_a4 = np.linspace(-2, 2, 60)[:, None]
y_a4 = 2.0 * X_a4[:, 0] + 0.5
qs_a4 = [0.9, 0.25]
curves_a4 = []
print("keep probabilities:", qs_a4)

▶ What you'll see: a simple one-feature regression problem where strong dropout is deliberately harsh.

In [ ]:
for q_a4 in qs_a4:
    np.random.seed(int(q_a4 * 1000))
    w_a4 = 0.0
    b_a4 = 0.0
    losses_a4 = []
    for epoch_a4 in range(120):
        losses_a4.append(float(np.mean((X_a4[:, 0] * w_a4 + b_a4 - y_a4) ** 2)))
        for i_a4 in range(X_a4.shape[0]):
            mask_a4 = 1.0 if np.random.rand() < q_a4 else 0.0
            xd_a4 = X_a4[i_a4, 0] * mask_a4 / q_a4
            err_a4 = xd_a4 * w_a4 + b_a4 - y_a4[i_a4]
            w_a4 -= 0.02 * err_a4 * xd_a4
            b_a4 -= 0.02 * err_a4
    curves_a4.append(losses_a4)
print("final losses:", [round(c_a4[-1], 3) for c_a4 in curves_a4])
assert curves_a4[0][-1] < curves_a4[1][-1]

▶ What you'll see: mild dropout fits the line better than aggressive dropout on this tiny problem.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(curves_a4[0], label="q=0.9 mild", color="teal")
plt.plot(curves_a4[1], label="q=0.25 aggressive", color="crimson")
plt.yscale("log")
plt.title("Advanced 4: too much dropout can underfit")
plt.xlabel("epoch")
plt.ylabel("MSE, log scale")
plt.legend()
plt.show()

▶ What you'll see: the aggressive-dropout curve settles at a higher error.

In [ ]:
ratio_a4 = curves_a4[1][-1] / curves_a4[0][-1]
print("aggressive/mild final-loss ratio:", round(ratio_a4, 2))
assert ratio_a4 > 1.0
plt.figure(figsize=(4, 3))
plt.bar(["mild q=0.9", "aggressive q=0.25"], [curves_a4[0][-1], curves_a4[1][-1]], color=["teal", "crimson"])
plt.title("Advanced 4: final underfit comparison")
plt.ylabel("final MSE")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the aggressive keep probability leaves a larger final training error.

👀 Takeaway: dropout is regularization, so too much of it can reduce capacity and slow learning.

### Advanced 5 — Combine dropout with L2 weight decay

**Goal.** Train the same small model with dropout, L2 shrinkage, or both, because regularizers can stack but also change the learned weight scale. We build it in 5 steps.

In [ ]:
np.random.seed(25)
X_a5 = np.random.normal(size=(80, 3))
true_w_a5 = np.array([1.5, -2.0, 0.5])
y_a5 = X_a5 @ true_w_a5 + 0.1 * np.random.normal(size=80)
settings_a5 = [(1.0, 0.0, "none"), (0.7, 0.0, "dropout"), (1.0, 0.05, "L2"), (0.7, 0.05, "both")]
print("settings:", [s_a5[2] for s_a5 in settings_a5])

▶ What you'll see: four regularization settings on the same synthetic regression data.

In [ ]:
losses_a5 = []
norms_a5 = []
for q_a5, lam_a5, name_a5 in settings_a5:
    np.random.seed(len(name_a5) * 10)
    w_a5 = np.zeros(3)
    b_a5 = 0.0
    for epoch_a5 in range(250):
        for i_a5 in range(X_a5.shape[0]):
            mask_a5 = (np.random.rand(3) < q_a5).astype(float) if q_a5 < 1.0 else np.ones(3)
            xd_a5 = X_a5[i_a5] * mask_a5 / q_a5
            err_a5 = float(xd_a5 @ w_a5 + b_a5 - y_a5[i_a5])
            w_a5 -= 0.02 * (err_a5 * xd_a5 + lam_a5 * w_a5)
            b_a5 -= 0.02 * err_a5
    pred_a5 = X_a5 @ w_a5 + b_a5
    losses_a5.append(float(np.mean((pred_a5 - y_a5) ** 2)))
    norms_a5.append(float(np.linalg.norm(w_a5)))
print("MSE:", np.round(losses_a5, 3))
print("weight norms:", np.round(norms_a5, 3))
assert min(losses_a5) < 0.05

▶ What you'll see: all settings learn useful weights, with different training errors and norms.

In [ ]:
labels_a5 = [s_a5[2] for s_a5 in settings_a5]
plt.figure(figsize=(5, 3))
plt.bar(labels_a5, losses_a5, color="steelblue")
plt.title("Advanced 5: regularizer comparison")
plt.ylabel("training MSE")
plt.show()

▶ What you'll see: stronger regularization may slightly raise training error.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(labels_a5, norms_a5, color="darkorange")
plt.title("Advanced 5: learned weight scale")
plt.ylabel("||w||₂")
plt.show()

▶ What you'll see: L2 directly shrinks weights, while dropout changes scale through noisy feature usage.

In [ ]:
best_idx_a5 = int(np.argmin(losses_a5))
print("lowest training MSE setting:", labels_a5[best_idx_a5])
assert labels_a5[best_idx_a5] in labels_a5
plt.figure(figsize=(5, 3))
plt.scatter(norms_a5, losses_a5, color="purple")
for i_a5, label_a5 in enumerate(labels_a5):
    plt.text(norms_a5[i_a5], losses_a5[i_a5], label_a5)
plt.title("Advanced 5: norm-error tradeoff")
plt.xlabel("weight norm")
plt.ylabel("training MSE")
plt.show()

▶ What you'll see: different regularizers occupy different points on the error-versus-size plot.

👀 Takeaway: dropout and L2 both control capacity, but dropout does it through stochastic subnetworks while L2 directly penalizes weight magnitude.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Randomly removing activations or weights makes units stop relying on fragile co-adaptations.

Dropout masks hidden activations; DropConnect masks weights. Inverted scaling keeps expected activation size stable while injecting training noise. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def split_scale(X, y):
    if len(y) > 300:
        x_small, _, y_small, _ = train_test_split(X, y, train_size=300, random_state=6, stratify=y)
    else:
        x_small = X
        y_small = y
    x_tr, x_te, y_tr, y_te = train_test_split(x_small, y_small, test_size=0.4, random_state=0, stratify=y_small)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=1, keepdims=True)


def relu(z):
    return np.maximum(z, 0.0)


def init_weights(n_in, n_hidden, n_out, mode, seed):
    rng = np.random.default_rng(seed)
    if mode == "xavier":
        scale1 = math.sqrt(2.0 / (n_in + n_hidden))
        scale2 = math.sqrt(2.0 / (n_hidden + n_out))
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "he":
        scale1 = math.sqrt(2.0 / n_in)
        scale2 = math.sqrt(2.0 / n_hidden)
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "tiny":
        W1 = rng.normal(0.0, 0.01, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.01, size=(n_hidden, n_out))
    elif mode == "large":
        W1 = rng.normal(0.0, 2.0, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 2.0, size=(n_hidden, n_out))
    elif mode == "orthogonal":
        Q1, _ = np.linalg.qr(rng.normal(size=(n_in, max(n_in, n_hidden))))
        Q2, _ = np.linalg.qr(rng.normal(size=(n_hidden, max(n_hidden, n_out))))
        W1 = Q1[:, :n_hidden]
        W2 = Q2[:, :n_out]
    else:
        W1 = rng.normal(0.0, 0.1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.1, size=(n_hidden, n_out))
    b1 = np.zeros(n_hidden)
    b2 = np.zeros(n_out)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward(params, X, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    W1 = params["W1"]
    if dropconnect_p > 0.0 and rng is not None:
        keep_w = 1.0 - dropconnect_p
        mask_w = rng.binomial(1, keep_w, size=W1.shape) / keep_w
        W1 = W1 * mask_w
    z1 = X @ W1 + params["b1"]
    h1 = relu(z1)
    mask = None
    if dropout_p > 0.0 and rng is not None:
        keep = 1.0 - dropout_p
        mask = rng.binomial(1, keep, size=h1.shape) / keep
        h1 = h1 * mask
    logits = h1 @ params["W2"] + params["b2"]
    return z1, h1, logits, mask


def loss_and_grads(params, X, y, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    classes = params["b2"].shape[0]
    z1, h1, logits, mask = forward(params, X, dropout_p, rng, dropconnect_p)
    probs = softmax(logits)
    target = one_hot(y, classes)
    loss = -np.mean(np.sum(target * np.log(probs + 1e-12), axis=1))
    dlogits = (probs - target) / len(y)
    dW2 = h1.T @ dlogits
    db2 = np.sum(dlogits, axis=0)
    dh1 = dlogits @ params["W2"].T
    if mask is not None:
        dh1 = dh1 * mask
    dz1 = dh1 * (z1 > 0.0)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return loss, grads


def predict(params, X):
    _, _, logits, _ = forward(params, X)
    return np.argmax(logits, axis=1)


def eval_loss(params, X, y):
    classes = params["b2"].shape[0]
    _, _, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    return -float(np.mean(np.sum(target * np.log(probs + 1e-12), axis=1)))


def vector_norm(params):
    total = 0.0
    for value in params.values():
        total += float(np.sum(value * value))
    return math.sqrt(total)


def kfac_precondition_grads(params, X, y, grads, damping):
    classes = params["b2"].shape[0]
    z1, h1, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    dlogits = (probs - target) / len(y)
    dh1 = dlogits @ params["W2"].T
    dz1 = dh1 * (z1 > 0.0)
    out = {key: value.copy() for key, value in grads.items()}
    A1 = X.T @ X / len(y) + damping * np.eye(X.shape[1])
    S1 = dz1.T @ dz1 / len(y) + damping * np.eye(dz1.shape[1])
    A2 = h1.T @ h1 / len(y) + damping * np.eye(h1.shape[1])
    S2 = dlogits.T @ dlogits / len(y) + damping * np.eye(dlogits.shape[1])
    out["W1"] = np.linalg.solve(A1, grads["W1"]) @ np.linalg.inv(S1)
    out["W2"] = np.linalg.solve(A2, grads["W2"]) @ np.linalg.inv(S2)
    return out


def apply_update(params, grads, state, method, lr, t, config):
    beta1 = config.get("beta1", 0.9)
    beta2 = config.get("beta2", 0.999)
    eps = config.get("eps", 1e-8)
    mu = config.get("momentum", 0.0)
    weight_decay = config.get("weight_decay", 0.0)
    for key in params:
        grad = grads[key]
        if method == "sgd":
            update = -lr * grad
        elif method == "momentum":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        elif method == "adagrad":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc += grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "rmsprop":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc *= beta2
            acc += (1.0 - beta2) * grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "adam" or method == "adamw":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            m_hat = m / (1.0 - beta1 ** t)
            v_hat = v / (1.0 - beta2 ** t)
            update = -lr * m_hat / (np.sqrt(v_hat) + eps)
            if method == "adamw" and key.startswith("W"):
                update -= lr * weight_decay * params[key]
        elif method == "lion":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            blended = beta1 * m + (1.0 - beta1) * grad
            update = -lr * np.sign(blended)
            m *= beta2
            m += (1.0 - beta2) * grad
        elif method == "lamb":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            raw = m / (np.sqrt(v) + eps)
            if key.startswith("W"):
                raw += weight_decay * params[key]
            ratio = np.linalg.norm(params[key]) / (np.linalg.norm(raw) + eps)
            ratio = float(np.clip(ratio, 0.1, 10.0))
            update = -lr * ratio * raw
        elif method == "nesterov":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        else:
            update = -lr * grad
        if weight_decay > 0.0 and method not in ["adamw", "lamb"] and key.startswith("W"):
            update -= lr * weight_decay * params[key]
        params[key] += update


def train_mlp(x_tr, y_tr, x_te, y_te, method="sgd", init="he", epochs=12, lr=0.05, hidden=8, batch_size=None, config=None, dropout_p=0.0, dropconnect_p=0.0, seed=0, early_patience=None):
    if config is None:
        config = {}
    classes = int(np.max(y_tr)) + 1
    params = init_weights(x_tr.shape[1], hidden, classes, init, seed)
    state = {}
    rng = np.random.default_rng(seed + 100)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "norm": []}
    best_loss = float("inf")
    best_params = None
    bad_epochs = 0
    n = len(y_tr)
    if batch_size is None:
        batch_size = n
    for epoch in range(1, epochs + 1):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            if method == "nesterov":
                lookahead = {}
                for key in params:
                    velocity = state.setdefault("v_" + key, np.zeros_like(params[key]))
                    lookahead[key] = params[key].copy()
                    params[key] += config.get("momentum", 0.9) * velocity
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
                for key in params:
                    params[key] = lookahead[key]
            else:
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
            if method == "kfac":
                grads = kfac_precondition_grads(params, x_tr[idx], y_tr[idx], grads, config.get("damping", 0.03))
                apply_update(params, grads, state, "sgd", lr, epoch, config)
            else:
                apply_update(params, grads, state, method, lr, epoch, config)
        train_loss = eval_loss(params, x_tr, y_tr)
        val_loss = eval_loss(params, x_te, y_te)
        train_acc = accuracy_score(y_tr, predict(params, x_tr))
        val_acc = accuracy_score(y_te, predict(params, x_te))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["norm"].append(vector_norm(params))
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = {key: value.copy() for key, value in params.items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if early_patience is not None and bad_epochs >= early_patience:
            params = best_params
            break
    return params, history


def run_component_ladder(variants, metric="accuracy", epochs=12, hidden=16):
    rows = []
    histories = {}
    artifacts = {}
    for rung_index, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        histories[name] = {}
        artifacts[name] = {}
        for variant in variants:
            params, hist = train_mlp(
                x_tr,
                y_tr,
                x_te,
                y_te,
                method=variant.get("method", "sgd"),
                init=variant.get("init", "he"),
                epochs=variant.get("epochs", epochs),
                lr=variant.get("lr", 0.05),
                hidden=hidden,
                batch_size=variant.get("batch_size"),
                config=variant.get("config", {}),
                dropout_p=variant.get("dropout_p", 0.0),
                dropconnect_p=variant.get("dropconnect_p", 0.0),
                seed=variant.get("seed", 10 + rung_index),
                early_patience=variant.get("early_patience"),
            )
            preds = predict(params, x_te)
            acc = accuracy_score(y_te, preds)
            val_loss = eval_loss(params, x_te, y_te)
            value = acc if metric == "accuracy" else val_loss
            rows.append({"rung": name, "variant": variant["name"], "accuracy": acc, "loss": val_loss, "metric": value})
            histories[name][variant["name"]] = hist
            artifacts[name][variant["name"]] = (x_te, y_te, preds)
    return rows, histories, artifacts


def print_table(rows, metric_name):
    print(f"{'rung':34s} {'variant':18s} {metric_name:>10s} {'acc':>8s} {'loss':>8s}")
    for row in rows:
        print(f"{row['rung'][:34]:34s} {row['variant'][:18]:18s} {row['metric']:10.3f} {row['accuracy']:8.3f} {row['loss']:8.3f}")


def plot_results(rows, histories, artifacts, metric_name, best_variant):
    rung_names = list(histories.keys())
    fig, axes = plt.subplots(2, len(rung_names), figsize=(3.2 * len(rung_names), 6.4))
    for col, rung in enumerate(rung_names):
        x_te, y_te, preds = artifacts[rung][best_variant]
        if x_te.shape[1] > 2:
            shown = PCA(n_components=2, random_state=0).fit_transform(x_te)
        else:
            shown = x_te[:, :2]
        axes[0, col].scatter(shown[:, 0], shown[:, 1], c=preds, s=12, cmap="tab10", alpha=0.85)
        axes[0, col].set_title(rung.split("(")[0].strip())
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])
        hist = histories[rung][best_variant]
        curve_key = "val_acc" if metric_name == "accuracy" else "val_loss"
        axes[1, col].plot(hist[curve_key], label=best_variant)
        axes[1, col].set_xlabel("epoch")
        axes[1, col].set_title(metric_name)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 3.5))
    variants = sorted({row["variant"] for row in rows})
    for variant in variants:
        vals = [row["metric"] for row in rows if row["variant"] == variant]
        ax.plot(range(1, len(vals) + 1), vals, marker="o", label=variant)
    ax.set_xticks(range(1, len(rung_names) + 1))
    ax.set_xticklabels([f"D{i}" for i in range(1, len(rung_names) + 1)])
    ax.set_ylabel(metric_name)
    ax.set_title("Same ladder, component varied")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## The concept, built once: inverted dropout

The lesson formula is
$$\tilde h=\frac{m\odot h}{q},\qquad m_i\sim\mathrm{Bernoulli}(q).$$
The lesson's local update has $\eta=0.070$, $g=1.950$, and $\theta=2.000$, so the plain move is $1.8635$.

In [ ]:

def regularizer_sweep(dropout_p):
    h = np.array([2.0, 4.0, 0.0, 6.0])
    q = 1.0 - dropout_p
    mask = np.array([1.0, 0.0, 1.0, 1.0])
    dropped = mask * h / q
    return dropped

dropped = regularizer_sweep(0.25)
plain_theta = 2.0 - 0.070 * 1.950
print(dropped, plain_theta)
assert np.allclose(dropped, np.array([2.6666666667, 0.0, 0.0, 8.0]))
assert abs(plain_theta - 1.8635) < 1e-12


DropConnect applies the same Bernoulli idea to weights instead of activations. The expected weighted signal is preserved by dividing by the keep probability.

In [ ]:

W = np.array([[1.0, -2.0], [3.0, 4.0]])
mask = np.array([[1.0, 0.0], [1.0, 1.0]])
q = 0.75
masked_W = W * mask / q
print(masked_W)
assert abs(masked_W[1, 1] - 5.333333333333333) < 1e-12


## The dataset ladder

Every topic uses the same `clf_digits_ladder()` and the same small MLP. Only the named optimizer or regularization component changes from variant to variant.

In [ ]:

rungs = clf_digits_ladder()
for name, X, y in rungs:
    classes = np.unique(y)
    print(f"{name:38s} shape={X.shape} classes={len(classes)} sample_y={y[:8].tolist()}")
print("D1 sample X:")
print(rungs[0][1])


## Run the same method across D1-D5

The architecture, splits, scaling, and seed policy stay fixed. The table reports one comparable metric per rung.

In [ ]:

variants = [
    {"name": "no dropout", "method": "adam", "lr": 0.015},
    {"name": "dropout p=.25", "method": "adam", "lr": 0.015, "dropout_p": 0.25},
    {"name": "DropConnect p=.15", "method": "adam", "lr": 0.015, "dropconnect_p": 0.15},
]

rows, histories, artifacts = run_component_ladder(variants, metric="accuracy", epochs=10, hidden=8)
print_table(rows, "accuracy")


## Results visualization

Top row: small multiples of held-out predictions. Bottom row: validation curves for the highlighted variant, followed by the component summary curve.

In [ ]:

plot_results(rows, histories, artifacts, "accuracy", "dropout p=.25")


## Pitfall on D5: dropout is not automatically better

Dropout can hurt underfit small data but help the noisy D5 gap. The fix is to tune the keep probability instead of treating randomness as magic.

In [ ]:

name, X, y = clf_digits_ladder()[0]
x_tr, x_te, y_tr, y_te = split_scale(X, y)
_, d1_no = train_mlp(x_tr, y_tr, x_te, y_te, method="adam", lr=0.02, dropout_p=0.0, epochs=10, seed=111)
_, d1_drop = train_mlp(x_tr, y_tr, x_te, y_te, method="adam", lr=0.02, dropout_p=0.5, epochs=10, seed=111)
name, X, y = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X, y)
_, d5_tuned = train_mlp(x_tr, y_tr, x_te, y_te, method="adam", lr=0.015, dropout_p=0.25, epochs=10, seed=111)
print("D1 no/drop", round(d1_no["val_acc"][-1], 3), round(d1_drop["val_acc"][-1], 3))
print("D5 tuned dropout", round(d5_tuned["val_acc"][-1], 3))
assert d5_tuned["val_acc"][-1] > 0.2


## Evaluate it + Practice

- Main metric: held-out accuracy on every D1-D5 rung, compared with a no-skill baseline near random guessing.
- Sanity check: D1 XOR should improve above chance once the hidden ReLU layer is active.
- Ablation: set dropout probability to zero or to an extreme 0.8; the metric should drop or the curve should become less stable.
- Failure signals: exploding loss, flat accuracy near chance, or a D5 train/validation gap that moves in opposite directions.
- Reproducibility: seeds are fixed and the ladder uses sklearn-bundled data only.

Practice 1: Change one hyperparameter in the strongest variant and rerun the summary curve.

Practice 2: Add a new diagnostic printout that distinguishes train accuracy from validation accuracy.

Practice 3: Explain why D5 is harder than D1 using the table and one plotted curve.